In [70]:
import pandas as pd
import sqlite3
from datetime import datetime

from zoneinfo import ZoneInfo

# Test case: Payments reconciliation — June 2026

## Context

Lumo Wellness sells app subscriptions through five payment providers: PayPal (separate US and EU accounts), dLocal (Latin America), Adyen, and Google Play.

In [71]:
# PayPal US account activity export
df_paypal_us_activity_20260601_20260703 = pd.read_csv("raw/paypal_us_activity_20260601_20260703.csv") 
df_paypal_us_activity_20260601_20260703.columns = df_paypal_us_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_us_activity_20260601_20260703["datetime_utc"] = df_paypal_us_activity_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['date']} {row['time']}").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_us_activity_20260601_20260703.head()


,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/01/2026,07:12:11,GMT,Taylor Brooks,7FTLSR4XAT5TKVLKF,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500884,2026-06-01 07:12:11+00:00
1,06/01/2026,07:54:32,GMT,Morgan Hayes,7R82A7SLSXJAD72J1,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500132,2026-06-01 07:54:32+00:00
2,06/01/2026,07:59:18,GMT,Drew Bennett,S1VNZT7ACC5E6HNYD,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500591,2026-06-01 07:59:18+00:00
3,06/01/2026,08:03:29,GMT,Riley Carter,BTDR7VQTQ8G4GTMGS,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500479,2026-06-01 08:03:29+00:00
4,06/01/2026,08:16:21,GMT,Casey Nguyen,NFQZ142PMP4P1BC2Q,NaN,Website Payment,Completed,USD,24.99,-1.36,23.63,ORD-500005,2026-06-01 08:16:21+00:00


In [72]:
# PayPal EU account activity export
# semicolon, cp1252 (Windows-1252)
df_paypal_eu_activity_20260601_20260703 = pd.read_csv("raw/paypal_eu_activity_20260601_20260703.csv", sep=";", encoding="cp1252") 
df_paypal_eu_activity_20260601_20260703.columns = df_paypal_eu_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_paypal_eu_activity_20260601_20260703["datetime_utc"] = df_paypal_eu_activity_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['date']} {row['time']}").tz_localize(ZoneInfo(row["time_zone"])).tz_convert("UTC")
    , axis=1)
df_paypal_eu_activity_20260601_20260703.head()

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id,datetime_utc
0,06/03/2026,12:20:00,GMT,Jürgen Müller,N2D9QMQNH8VUEES5X,NaN,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507782,2026-06-03 12:20:00+00:00
1,01/06/2026,08:17:07,GMT,Céline Marchand,ZEW9RTFHKTBFQT5E3,NaN,Website Payment,Completed,GBP,"8,99","-0,56","8,43",ORD-502669,2026-01-06 08:17:07+00:00
2,01/06/2026,08:35:02,GMT,José García,0DRQ0928NVSA52MA3,NaN,Website Payment,Denied,GBP,"4,49","0,00","0,00",ORD-503463,2026-01-06 08:35:02+00:00
3,01/06/2026,08:48:13,GMT,Nuño Ibáñez,P54742W2ULA8YDR0L,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502556,2026-01-06 08:48:13+00:00
4,01/06/2026,09:07:25,GMT,Søren Kjær,JR1SUR8G07W17ZPNR,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502537,2026-01-06 09:07:25+00:00


In [73]:
# Adyen payment accounting report
df_adyen_payment_accounting_20260601_20260703 = pd.read_csv("raw/adyen_payment_accounting_20260601_20260703.csv") 
df_adyen_payment_accounting_20260601_20260703.columns = df_adyen_payment_accounting_20260601_20260703.columns.str.lower().str.replace(" ", "_")

df_adyen_payment_accounting_20260601_20260703["datetime_utc"] = df_adyen_payment_accounting_20260601_20260703.apply(
    lambda row: pd.Timestamp(f"{row['creation_date']}").tz_localize(ZoneInfo(row["timezone"])).tz_convert("UTC")
    , axis=1)
df_adyen_payment_accounting_20260601_20260703.head()

,company_account,merchant_account,psp_reference,merchant_reference,record_type,creation_date,timezone,gross_currency,gross_debit,gross_credit,commission,markup,net_currency,net_debit,net_credit,batch_number,datetime_utc
0,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Authorised,2026-06-01 09:13:28,Europe/Amsterdam,USD,NaN,24.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:13:28+00:00
1,LumoGroup,LumoECOM_EU,8620000000000500,ORD-504450,Authorised,2026-06-01 09:15:43,Europe/Amsterdam,EUR,NaN,9.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:15:43+00:00
2,LumoGroup,LumoECOM_EU,8620000000000115,ORD-504065,Authorised,2026-06-01 09:18:57,Europe/Amsterdam,EUR,NaN,24.99,NaN,NaN,EUR,NaN,NaN,23,2026-06-01 07:18:57+00:00
3,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Settled,2026-06-01 09:36:28,Europe/Amsterdam,USD,NaN,24.99,0.47,0.15,USD,NaN,24.37,23,2026-06-01 07:36:28+00:00
4,LumoGroup,LumoAPP_US,8620000000000575,ORD-504525,Authorised,2026-06-01 09:41:36,Europe/Amsterdam,USD,NaN,9.99,NaN,NaN,USD,NaN,NaN,23,2026-06-01 07:41:36+00:00


In [8]:
# dLocal transactions export
df_dlocal_transactions_20260601_20260703 = pd.read_csv("raw/dlocal_transactions_20260601_20260703.csv") 
df_dlocal_transactions_20260601_20260703.head()

,transaction_id,invoice_id,transaction_type,status,created_at_utc,country,currency,currency_exponent,local_amount,fx_rate,usd_amount,fee_usd
0,DL-90001309,ORD-507781,PAYMENT,PAID,2026-06-01 00:05:00,MX,MXN,2,12900,18.1548,7.11,0.32
1,DL-90001242,ORD-506937,PAYMENT,REJECTED,2026-06-01 07:25:52,CO,COP,2,19990000,3984.0637,50.17,2.26
2,DL-90000404,ORD-506099,PAYMENT,PAID,2026-06-01 08:17:24,MX,MXN,2,12900,18.1772,7.10,0.32
3,DL-90000750,ORD-506445,PAYMENT,PAID,2026-06-01 08:27:11,CO,COP,2,2990000,3984.0637,7.50,0.34
4,DL-90001024,ORD-506719,PAYMENT,PAID,2026-06-01 08:29:13,BR,BRL,2,3490,5.4195,6.44,0.29


In [74]:
# Google Play earnings report, June
# line 1 is a report header, data starts on line 2
df_google_play_earnings_202606 = pd.read_csv("raw/google_play_earnings_202606.csv", skiprows=1, header=0)
df_google_play_earnings_202606.columns = df_google_play_earnings_202606.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202606["datetime_utc"] = df_google_play_earnings_202606.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202606.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jun 1, 2026",00:28:46,Charge,sub_annual,US,USD,59.99,1.0,USD,59.99,2026-06-01 07:28:46+00:00
1,"Jun 1, 2026",00:28:46,Google fee,sub_annual,US,USD,-9.00,1.0,USD,-9.00,2026-06-01 07:28:46+00:00
2,"Jun 1, 2026",00:53:08,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 07:53:08+00:00
3,"Jun 1, 2026",00:53:08,Google fee,sub_monthly,US,USD,-1.50,1.0,USD,-1.50,2026-06-01 07:53:08+00:00
4,"Jun 1, 2026",01:35:01,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99,2026-06-01 08:35:01+00:00


In [75]:
# Google Play earnings report, July 1–3
df_google_play_earnings_202607_partial = pd.read_csv("raw/google_play_earnings_202607_partial.csv", skiprows=1, header=0) 
df_google_play_earnings_202607_partial.columns = df_google_play_earnings_202607_partial.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

df_google_play_earnings_202607_partial["datetime_utc"] = df_google_play_earnings_202607_partial.apply(
    lambda row: pd.Timestamp(f"{row['transaction_date']} {row['transaction_time']}").tz_localize(ZoneInfo("America/Los_Angeles")).tz_convert("UTC")
    , axis=1)
df_google_play_earnings_202607_partial.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency,datetime_utc
0,"Jul 1, 2026",02:40:10,Charge refund,sub_monthly,DE,EUR,-9.99,1.085329,USD,-10.84,2026-07-01 09:40:10+00:00
1,"Jul 1, 2026",03:50:07,Charge,sub_monthly,DE,EUR,9.99,1.085329,USD,10.84,2026-07-01 10:50:07+00:00
2,"Jul 1, 2026",03:50:07,Google fee,sub_monthly,DE,EUR,-1.50,1.085329,USD,-1.63,2026-07-01 10:50:07+00:00
3,"Jul 1, 2026",05:43:32,Charge,sub_monthly,US,USD,9.99,1.000000,USD,9.99,2026-07-01 12:43:32+00:00
4,"Jul 1, 2026",05:43:32,Google fee,sub_monthly,US,USD,-1.50,1.000000,USD,-1.50,2026-07-01 12:43:32+00:00


payment_engine_log.csv is an extract from the payment backend — the engine that initiates and tracks every transaction attempt, applies FX conversion, and is treated internally as the source of truth. The other files are data exports from each payment provider’s admin panel — it’s what the PSPs themselves say happened.

In [3]:
# Payment backend (engine) log extract, all providers
df_payment_engine_log = pd.read_csv("raw/payment_engine_log.csv") 
df_payment_engine_log.head()

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z


In [18]:
# Daily FX rates to USD (usd_rate = USD per 1 unit of currency)
df_fx_rates = pd.read_csv("raw/fx_rates.csv") 
df_fx_rates.head()

,rate_date,currency,usd_rate,published_at
0,2026-05-25,ARS,0.000839,2026-05-25T16:00:00Z
1,2026-05-25,BRL,0.184871,2026-05-25T16:00:00Z
2,2026-05-25,CLP,0.001060,2026-05-25T16:00:00Z
3,2026-05-25,COP,0.000248,2026-05-25T16:00:00Z
4,2026-05-25,EUR,1.093525,2026-05-25T16:00:00Z


In [19]:
# Contracted provider fees, where we have a contract on file
df_fee_schedule = pd.read_csv("raw/fee_schedule.csv") 
df_fee_schedule.head()

,psp,account,valid_from,percent_fee,fixed_fee,fixed_fee_currency
0,adyen,LumoECOM_EU,2026-01-01,2.5,0.0,NaN
1,adyen,LumoAPP_US,2026-01-01,2.5,0.0,NaN
2,google_play,LumoPlayMain,2026-01-01,15.0,0.0,NaN


Every month, Finance Analyst must prove that the payment backend and the provider exports tell the same story, and explain every difference in dollars if it’s not the case. That is your task, for the reporting period June 2026.

The engine-log extract and the provider exports intentionally cover a few days around the month boundary as well.


In [76]:
# connect to a database
conn = sqlite3.connect("db_simple_life.db")

df_payment_engine_log.to_sql('payment_engine_log', conn, if_exists="replace", index=False)

df_paypal_us_activity_20260601_20260703.to_sql('paypal_us_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_paypal_eu_activity_20260601_20260703.to_sql('paypal_eu_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_adyen_payment_accounting_20260601_20260703.to_sql('adyen_payment_accounting_20260601_20260703', conn, if_exists="replace", index=False)
df_dlocal_transactions_20260601_20260703.to_sql('dlocal_transactions_20260601_20260703', conn, if_exists="replace", index=False)
df_google_play_earnings_202606.to_sql('google_play_earnings_202606', conn, if_exists="replace", index=False)
df_google_play_earnings_202607_partial.to_sql('google_play_earnings_202607_partial', conn, if_exists="replace", index=False)
df_fx_rates.to_sql('fx_rates', conn, if_exists="replace", index=False)
df_fee_schedule.to_sql('fee_schedule', conn, if_exists="replace", index=False)


3

## Your task
- Reconcile each provider export against the payment engine log for June 2026. Quantify all discrepancies — in row counts and in USD.
- Classify every discrepancy by root cause, using your own taxonomy.
- Deliver a summary at provider × cause grain: row count, absolute USD impact, signed USD impact, and each cause’s share of the total discrepancy in %.
- Where fee_schedule.csv is applicable to  a provider, reconcile the reported fees against the contracted rates.
- Report the total discrepancy and show what remains unexplained after your classification (the residual). A correct analysis explains all the discrepancies.

## Evaluation notes
- Use DB of your choice (but tell us which), we assess the logic, not the dialect.
- When the data is ambiguous, make a call and document it. Documented assumptions count in your favor.
- Every dollar of difference between the payment backend and the providers is either matched, explained with a cause, or explicitly listed as unexplained — and the three add up.
- The reconciliation summary should read like something you could hand to a CFO: numbers first, caveats and issues stated.

## What to hand back
- Your SQL
- The summary from step 3, as a query result (CSV).
- A short written note: your assumptions, data-quality findings, and action items you would take to fix upstream.


In [78]:
# query0 = """
# SELECT *
# -- FROM payment_engine_log -- +
# -- FROM paypal_us_activity_20260601_20260703 -- +
# -- FROM paypal_eu_activity_20260601_20260703 -- +
# -- FROM adyen_payment_accounting_20260601_20260703 -- +
# -- FROM dlocal_transactions_20260601_20260703 -- +
# FROM google_play_earnings_202606 -- +
# -- FROM google_play_earnings_202607_partial -- +
# -- FROM fx_rates -- +
# -- FROM fee_schedule -- +
# LIMIT 100;
# """
# result0 = pd.read_sql(query0, conn)
# result0

In [54]:
query = """
--WITH adyen as (
--    SELECT 
--    FROM adyen_payment_accounting_20260601_20260703
--    WHERE
--)

SELECT 
    -- sku?? status?? operation_type??
    psp, 
    country, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
GROUP BY 1,2,3
LIMIT 100;
"""

In [55]:
result = pd.read_sql(query, conn)
result

,psp,country,currency,amount_local,amount_usd
0,adyen,DE,EUR,10269.94,11174.15
1,adyen,FR,EUR,4924.89,5358.69
2,adyen,GB,GBP,4941.21,6383.63
3,adyen,NL,EUR,6646.49,7236.50
4,adyen,US,USD,6941.67,6941.67
5,dlocal,AR,ARS,1300490.50,1086.98
6,dlocal,BR,BRL,40246.37,7491.96
7,dlocal,CL,CLP,2057005.00,2184.89
8,dlocal,CO,COP,7891100.00,201797.80
9,dlocal,MX,MXN,77300.00,5019.47


- payment_engine_log - created_at_utc
- paypal (eu, us) - GMT
- adyen - Europe/Amsterdam
- dlocal - created_at_utc
- google play - All dates and times are in America/Los_Angeles

### Paypal EU

In [110]:
query_paypal = """
WITH paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(pp.gross) as gross
        , sum(pp.fee) as fee
        , sum(pp.net) as net
        , sum(pp.gross * fx.usd_rate) as gross_usd
        , sum(pp.net * fx.usd_rate) as net_usd
    FROM paypal_eu_activity_20260601_20260703 pp
    LEFT JOIN fx_rates fx on fx.rate_date = date(pp.datetime_utc) and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal = pd.read_sql(query_paypal, conn)
result_paypal

,type,status,currency,gross,fee,net,gross_usd,net_usd
0,Payment Reversal,Reversed,EUR,-59.0,0.0,-59.0,NaN,NaN
1,Refund,Refunded,EUR,-320.0,0.0,-320.0,-228.024148,-228.024148
2,Refund,Refunded,GBP,-76.0,0.0,-76.0,-41.541726,-41.541726
3,Website Payment,Completed,EUR,18271.0,-486.0,17785.0,11889.235896,11567.125558
4,Website Payment,Completed,GBP,2593.0,-16.0,2577.0,1997.733622,1986.090228
5,Website Payment,Denied,EUR,1263.0,0.0,0.0,777.145185,0.000000
6,Website Payment,Denied,GBP,318.0,0.0,0.0,257.811569,0.000000


In [114]:
query_paypal_eu_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'paypal_eu'
GROUP BY 1,2,3
LIMIT 100;
"""
result_paypal_eu_backend = pd.read_sql(query_paypal_eu_backend, conn)
result_paypal_eu_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,EUR,2253.81,2453.18
1,declined,SALE,GBP,506.73,655.31
2,settled,REFUND,EUR,329.84,358.63
3,settled,REFUND,GBP,101.45,131.19
4,settled,SALE,EUR,28775.99,31310.48
5,settled,SALE,GBP,4259.41,5502.35


All numbers are discrepancies

### PayPal US

In [116]:
query_paypal_us = """
WITH paypal as (
    SELECT 
        pp.type,
        pp.status,
        pp.currency				
        , sum(pp.gross) as gross
        , sum(pp.fee) as fee
        , sum(pp.net) as net
        -- , sum(pp.gross * fx.usd_rate) as gross_usd
        -- , sum(pp.net * fx.usd_rate) as net_usd
    FROM paypal_us_activity_20260601_20260703 pp
    LEFT JOIN fx_rates fx on fx.rate_date = date(pp.datetime_utc) and fx.currency = pp.currency	
    WHERE pp.datetime_utc >= '2026-06-01' AND pp.datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM paypal
LIMIT 100;
"""
result_paypal_us = pd.read_sql(query_paypal_us, conn)
result_paypal_us

,type,status,currency,gross,fee,net
0,Refund,Refunded,USD,-649.76,0.0,-649.76
1,Website Payment,Completed,USD,36731.05,-2209.3,34521.75
2,Website Payment,Denied,USD,2923.42,0.0,0.00


In [115]:
query_paypal_us_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'paypal_us'
GROUP BY 1,2,3
LIMIT 100;
"""
result_paypal_us_backend = pd.read_sql(query_paypal_us_backend, conn)
result_paypal_us_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,USD,2923.42,2923.42
1,pending,SALE,USD,24.99,24.99
2,settled,REFUND,USD,649.76,649.76
3,settled,SALE,USD,36716.05,36716.05


Discrepancies:
- settled	SALE: 36716.05 vs 	36731.05
- pending	SALE: 24.99	vs None

### Adyen

In [90]:
query_adyen = """
WITH adyen as (
    SELECT 
        aa.record_type,
        aa.gross_currency,
        aa.net_currency
        , sum(aa.gross_credit) as gross_credit
        , sum(aa.gross_debit) as gross_debit
        , sum(aa.commission) as commission
        , sum(aa.net_credit) as net_credit
        , sum(aa.net_debit) as net_debit
        , sum(aa.gross_credit * coalesce(fx.usd_rate, 1)) as gross_credit_usd
        , sum(aa.gross_debit * coalesce(fx.usd_rate, 1)) as gross_debit_usd
        , sum(aa.net_credit * coalesce(fx.usd_rate, 1)) as net_credit_usd
        , sum(aa.net_debit * coalesce(fx.usd_rate, 1)) as net_debit_usd
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates fx on fx.rate_date = date(aa.datetime_utc) and fx.currency = aa.gross_currency	
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM adyen
LIMIT 100;
"""
result_adyen = pd.read_sql(query_adyen, conn)
result_adyen

,record_type,gross_currency,net_currency,gross_credit,gross_debit,commission,net_credit,net_debit,gross_credit_usd,gross_debit_usd,net_credit_usd,net_debit_usd
0,Authorised,EUR,EUR,19389.92,NaN,NaN,NaN,NaN,20586.775699,NaN,NaN,NaN
1,Authorised,GBP,GBP,4607.37,NaN,NaN,NaN,NaN,5556.290110,NaN,NaN,NaN
2,Authorised,USD,USD,6276.94,NaN,NaN,NaN,NaN,6276.940000,NaN,NaN,NaN
3,Chargeback,EUR,EUR,NaN,89.97,NaN,NaN,89.97,NaN,94.771720,NaN,94.771720
4,Refunded,EUR,EUR,NaN,272.37,NaN,NaN,272.37,NaN,287.461817,NaN,287.461817
5,Refunded,GBP,GBP,NaN,30.98,NaN,NaN,30.98,NaN,33.662625,NaN,33.662625
6,Refunded,USD,USD,NaN,49.97,NaN,NaN,49.97,NaN,49.970000,NaN,49.970000
7,Refused,EUR,EUR,2159.04,NaN,NaN,NaN,NaN,2305.550701,NaN,NaN,NaN
8,Refused,GBP,GBP,302.86,NaN,NaN,NaN,NaN,354.014414,NaN,NaN,NaN
9,Refused,USD,USD,614.76,NaN,NaN,NaN,NaN,614.760000,NaN,NaN,NaN


In [88]:
query_adyen_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'adyen'
GROUP BY 1,2,3
LIMIT 100;
"""
result_adyen_backend = pd.read_sql(query_adyen_backend, conn)
result_adyen_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,EUR,2149.05,2339.09
1,declined,SALE,GBP,302.86,391.20
2,declined,SALE,USD,614.76,614.76
3,settled,CHARGEBACK,EUR,29.98,32.67
4,settled,REFUND,EUR,272.37,295.90
5,settled,REFUND,GBP,30.98,40.21
6,settled,REFUND,USD,49.97,49.97
7,settled,SALE,EUR,19389.92,21101.68
8,settled,SALE,GBP,4607.37,5952.22
9,settled,SALE,USD,6276.94,6276.94


In [92]:
query_adyen_reconciliation = """
WITH adyen_backend AS (
    SELECT 
        -- sku??
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'adyen'
    GROUP BY 1,2,3
),

adyen as (
    SELECT 
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'settled'
            WHEN aa.record_type = 'Chargeback' THEN 'settled'
            WHEN aa.record_type = 'Refunded' THEN 'settled'
            WHEN aa.record_type = 'Refused' THEN 'declined'
            -- WHEN aa.record_type = 'Settled' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.record_type = 'Authorised' THEN 'SALE'
            WHEN aa.record_type = 'Chargeback' THEN 'CHARGEBACK'
            WHEN aa.record_type = 'Refunded' THEN 'REFUND'
            WHEN aa.record_type = 'Refused' THEN 'SALE'
            -- WHEN aa.record_type = 'Settled' THEN 'SALE'
        END as operation_type,

        aa.gross_currency,
        aa.net_currency
        , sum(aa.gross_credit) as gross_credit
        , sum(aa.gross_debit) as gross_debit
        , sum(aa.commission) as commission
        , sum(aa.net_credit) as net_credit
        , sum(aa.net_debit) as net_debit
        , sum(aa.gross_credit * coalesce(fx.usd_rate, 1)) as gross_credit_usd
        , sum(aa.gross_debit * coalesce(fx.usd_rate, 1)) as gross_debit_usd
        , sum(aa.net_credit * coalesce(fx.usd_rate, 1)) as net_credit_usd
        , sum(aa.net_debit * coalesce(fx.usd_rate, 1)) as net_debit_usd
    FROM adyen_payment_accounting_20260601_20260703 aa
    LEFT JOIN fx_rates fx on fx.rate_date = date(aa.datetime_utc) and fx.currency = aa.gross_currency	
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT 
    back.*,
    -- adyen.record_type,
    adyen.gross_currency,
    adyen.gross_credit,
    adyen.gross_debit
FROM adyen_backend back
LEFT JOIN adyen ON adyen.operation_type = back.operation_type 
    AND adyen.status = back.status
    AND adyen.gross_currency = back.currency
"""
result_adyen_reconciliation = pd.read_sql(query_adyen_reconciliation, conn)
result_adyen_reconciliation

,status,operation_type,currency,amount_local,amount_usd,gross_currency,gross_credit,gross_debit
0,declined,SALE,EUR,2149.05,2339.09,EUR,2159.04,NaN
1,declined,SALE,GBP,302.86,391.20,GBP,302.86,NaN
2,declined,SALE,USD,614.76,614.76,USD,614.76,NaN
3,settled,CHARGEBACK,EUR,29.98,32.67,EUR,NaN,89.97
4,settled,REFUND,EUR,272.37,295.90,EUR,NaN,272.37
5,settled,REFUND,GBP,30.98,40.21,GBP,NaN,30.98
6,settled,REFUND,USD,49.97,49.97,USD,NaN,49.97
7,settled,SALE,EUR,19389.92,21101.68,EUR,19389.92,NaN
8,settled,SALE,GBP,4607.37,5952.22,GBP,4607.37,NaN
9,settled,SALE,USD,6276.94,6276.94,USD,6276.94,NaN


Only 2 dicrepancies: 
- declined + SALE + EUR -- 2149.05 vs 2159.04
- settled + CHARGEBACK + EUR -- 29.98 vs 89.97

### dLocal

In [98]:
query_dlocal = """
WITH dlocal as (
    SELECT 
        transaction_type,
        status,
        country,
        currency
        , sum(round(local_amount, 2) / 100) as local_amount
        , sum(usd_amount) as usd_amount
        , sum(usd_amount * fx_rate) as fx_
    FROM dlocal_transactions_20260601_20260703
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT *
FROM dlocal
LIMIT 100;
"""
result_dlocal = pd.read_sql(query_dlocal, conn)
result_dlocal

,transaction_type,status,country,currency,local_amount,usd_amount,fx_
0,PAYMENT,IN_MEDIATION,BR,BRL,34.90,6.45,3.489321e+01
1,PAYMENT,PAID,AR,ARS,1170500.00,978.41,1.170573e+06
2,PAYMENT,PAID,BR,BRL,35360.00,6517.64,3.536021e+04
3,PAYMENT,PAID,CL,CLP,18756.60,1992.40,1.875667e+06
4,PAYMENT,PAID,CO,COP,7291800.00,1797.63,7.293027e+06
5,PAYMENT,PAID,MX,MXN,70395.00,3883.39,7.040214e+04
6,PAYMENT,REJECTED,AR,ARS,115992.00,96.92,1.159904e+05
7,PAYMENT,REJECTED,BR,BRL,3901.50,720.13,3.901470e+03
8,PAYMENT,REJECTED,CL,CLP,1668.70,177.13,1.668615e+05
9,PAYMENT,REJECTED,CO,COP,599300.00,148.95,5.993538e+05


In [93]:
query_dlocal_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'dlocal'
GROUP BY 1,2,3
LIMIT 100;
"""
result_dlocal_backend = pd.read_sql(query_dlocal_backend, conn)
result_dlocal_backend

,status,operation_type,currency,amount_local,amount_usd
0,declined,SALE,ARS,115992.00,96.92
1,declined,SALE,BRL,3901.50,720.13
2,declined,SALE,CLP,166870.00,177.13
3,declined,SALE,COP,599300.00,148.95
4,declined,SALE,MXN,6834.00,377.29
5,settled,REFUND,ARS,13998.50,11.67
6,settled,REFUND,BRL,949.97,174.91
7,settled,REFUND,CLP,7485.00,7.94
8,settled,REFUND,MXN,329.00,18.11
9,settled,SALE,ARS,1170500.00,978.39


In [103]:
query_dlocal_reconciliation = """
WITH dlocal_backend AS (
    SELECT 
        -- sku??
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'dlocal'
    GROUP BY 1,2,3
),

dlocal as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'settled'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'declined'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'REFUND' AND aa.status = 'REFUNDED' THEN 'REFUND'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'REJECTED' THEN 'SALE'
            WHEN aa.transaction_type = 'PAYMENT' AND aa.status = 'PAID' THEN 'SALE'
        END as operation_type,

        aa.transaction_type,
        aa.status as dlocal_status,
        currency
        , sum(round(local_amount, 2) / 100) as local_amount
        , sum(usd_amount) as usd_amount
        , sum(usd_amount * fx_rate) as fx_
    FROM dlocal_transactions_20260601_20260703 aa
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    GROUP BY 1,2,3,4,5
)

SELECT 
    back.*,
    dlocal.transaction_type,
    dlocal.dlocal_status,
    dlocal.currency,
    dlocal.local_amount,
    dlocal.usd_amount
FROM dlocal_backend back
FULL JOIN dlocal ON dlocal.operation_type = back.operation_type 
    AND dlocal.status = back.status
    AND dlocal.currency = back.currency
"""
result_dlocal_reconciliation = pd.read_sql(query_dlocal_reconciliation, conn)
result_dlocal_reconciliation

,status,operation_type,currency,amount_local,amount_usd,transaction_type,dlocal_status,currency,local_amount,usd_amount
0,declined,SALE,ARS,115992.00,96.92,PAYMENT,REJECTED,ARS,115992.00,96.92
1,declined,SALE,BRL,3901.50,720.13,PAYMENT,REJECTED,BRL,3901.50,720.13
2,declined,SALE,CLP,166870.00,177.13,PAYMENT,REJECTED,CLP,1668.70,177.13
3,declined,SALE,COP,599300.00,148.95,PAYMENT,REJECTED,COP,599300.00,148.95
4,declined,SALE,MXN,6834.00,377.29,PAYMENT,REJECTED,MXN,6834.00,377.29
5,settled,REFUND,ARS,13998.50,11.67,REFUND,REFUNDED,ARS,13998.50,11.67
6,settled,REFUND,BRL,949.97,174.91,REFUND,REFUNDED,BRL,989.95,182.27
7,settled,REFUND,CLP,7485.00,7.94,REFUND,REFUNDED,CLP,74.85,7.94
8,settled,REFUND,MXN,329.00,18.11,REFUND,REFUNDED,MXN,329.00,18.11
9,settled,SALE,ARS,1170500.00,978.39,PAYMENT,PAID,ARS,1170500.00,978.41


dicrepancies:
- declined + SALE + CLP -- 166870.00 vs 1668.70 (usd ok)
- settled	REFUND	CLP	-- 7485.00 vs 74.85 (usd ok)
- settled	REFUND	BRL	-- 949.97 vs 989.95 (usd TOO)
- settled	SALE	ARS -- USD 978.39 vs 978.41
- settled	SALE	BRL	-- 35394.90	vs 35360.00	(usd TOO)
- settled	SALE	CLP	-- 1882650.00 vs 18756.60	(usd TOO)
- settled	SALE	COP	-- USD 201648.85 vs 1797.63
- settled	SALE	MXN	-- USD 4624.07 vs 3883.39
- PAYMENT	IN_MEDIATION	BRL -- there is no backend

### Google Play

In [104]:
query_googleplay = """
WITH googleplay_0 as (
    SELECT * FROM google_play_earnings_202606
    UNION ALL
    SELECT * FROM google_play_earnings_202607_partial
),

googleplay as (
    SELECT 
        transaction_type,
        buyer_currency,
        merchant_currency
        , sum(amount_buyer_currency) as amount_buyer_currency
        , sum(amount_merchant_currency) as amount_merchant_currency
    FROM googleplay_0
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3
)

SELECT *
FROM googleplay
LIMIT 100;
"""
result_googleplay = pd.read_sql(query_googleplay, conn)
result_googleplay

,transaction_type,buyer_currency,merchant_currency,amount_buyer_currency,amount_merchant_currency
0,Charge,BRL,USD,8806.70,1622.49
1,Charge,EUR,USD,3438.49,3745.96
2,Charge,GBP,USD,1963.95,2538.65
3,Charge,MXN,USD,17698.00,976.39
4,Charge,USD,USD,6046.85,6046.85
5,Charge refund,BRL,USD,-34.90,-6.44
6,Charge refund,EUR,USD,-79.96,-86.87
7,Charge refund,GBP,USD,-21.99,-28.54
8,Charge refund,MXN,USD,-799.00,-43.89
9,Charge refund,USD,USD,-19.97,-19.97


In [105]:
query_googleplay_backend = """
SELECT 
    -- sku??
    status, 
    operation_type, 
    currency
    , sum(amount_local) as amount_local
    , sum(amount_usd) as amount_usd
FROM payment_engine_log
WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
    and psp = 'google_play'
GROUP BY 1,2,3
LIMIT 100;
"""
result_googleplay_backend = pd.read_sql(query_googleplay_backend, conn)
result_googleplay_backend

,status,operation_type,currency,amount_local,amount_usd
0,settled,REFUND,BRL,34.90,6.44
1,settled,REFUND,EUR,79.96,86.87
2,settled,REFUND,GBP,21.99,28.54
3,settled,REFUND,MXN,799.00,43.89
4,settled,REFUND,USD,19.97,19.97
5,settled,SALE,BRL,8806.70,1622.49
6,settled,SALE,EUR,3438.49,3745.96
7,settled,SALE,GBP,1963.95,2538.65
8,settled,SALE,MXN,17698.00,976.39
9,settled,SALE,USD,6046.85,6046.85


In [108]:
query_googleplay_reconciliation = """
WITH googleplay_backend AS (
    SELECT 
        -- sku??
        status, 
        operation_type, 
        currency
        , sum(amount_local) as amount_local
        , sum(amount_usd) as amount_usd
    FROM payment_engine_log
    WHERE created_at_utc >= '2026-06-01' AND created_at_utc < '2026-07-01'
        and psp = 'google_play'
    GROUP BY 1,2,3
),

googleplay_0 as (
    SELECT * FROM google_play_earnings_202606
    UNION ALL
    SELECT * FROM google_play_earnings_202607_partial
),

googleplay as (
    SELECT 
        CASE 
            WHEN aa.transaction_type = 'Charge' THEN 'settled'
            WHEN aa.transaction_type = 'Charge refund' THEN 'settled'
        END as status,
        CASE 
            WHEN aa.transaction_type = 'Charge' THEN 'SALE'
            WHEN aa.transaction_type = 'Charge refund' THEN 'REFUND'
        END as operation_type,

        transaction_type,
        buyer_currency,
        merchant_currency
        , sum(amount_buyer_currency) as amount_buyer_currency
        , sum(amount_merchant_currency) as amount_merchant_currency

    FROM googleplay_0 aa
--     LEFT JOIN fx_rates fx on fx.rate_date = date(aa.datetime_utc) and fx.currency = aa.gross_currency	
    WHERE datetime_utc >= '2026-06-01' AND datetime_utc < '2026-07-01'
    GROUP BY 1,2,3,4
)

SELECT 
    back.*,
    googleplay.transaction_type,
    googleplay.amount_buyer_currency,
    googleplay.amount_merchant_currency
FROM googleplay_backend back
FULL JOIN googleplay ON googleplay.operation_type = back.operation_type 
    AND googleplay.status = back.status
    AND googleplay.buyer_currency = back.currency
"""
result_googleplay_reconciliation = pd.read_sql(query_googleplay_reconciliation, conn)
result_googleplay_reconciliation

,status,operation_type,currency,amount_local,amount_usd,transaction_type,amount_buyer_currency,amount_merchant_currency
0,settled,REFUND,BRL,34.90,6.44,Charge refund,-34.90,-6.44
1,settled,REFUND,EUR,79.96,86.87,Charge refund,-79.96,-86.87
2,settled,REFUND,GBP,21.99,28.54,Charge refund,-21.99,-28.54
3,settled,REFUND,MXN,799.00,43.89,Charge refund,-799.00,-43.89
4,settled,REFUND,USD,19.97,19.97,Charge refund,-19.97,-19.97
5,settled,SALE,BRL,8806.70,1622.49,Charge,8806.70,1622.49
6,settled,SALE,EUR,3438.49,3745.96,Charge,3438.49,3745.96
7,settled,SALE,GBP,1963.95,2538.65,Charge,1963.95,2538.65
8,settled,SALE,MXN,17698.00,976.39,Charge,17698.00,976.39
9,settled,SALE,USD,6046.85,6046.85,Charge,6046.85,6046.85


No discrepancies in both currencies

### Further...